# Evolvable Policies â€” GPU-Accelerated Notebook

Replicates **Thoma, Vassiliades & Michael (2026)**:  
starting from an empty policy + random weights, a NeSy organism discovers a hidden
symbolic policy through evolutionary search â€” **no gradient signal on the policy**.

This notebook replaces the pure-NumPy CNN with a **PyTorch CNN on T4 GPU**,
keeping the symbolic/evolutionary logic unchanged.

**Runtime â†’ Change runtime type â†’ T4 GPU**

## 1. Setup

In [ ]:
import os

REPO_URL = 'https://github.com/konstantinoslamp/NeuroSymbolic-Network-for-Handwritten-Symbol-Understanding.git'
REPO_DIR = '/content/neurosymbolic_mvp'

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

# Clear Python bytecode cache so updated .py files are always used
import subprocess
subprocess.run(['find', '.', '-name', '*.pyc', '-delete'], capture_output=True)
subprocess.run(['find', '.', '-name', '__pycache__', '-type', 'd', '-exec', 'rm', '-rf', '{}', '+'], capture_output=True)

# Print commit hash so we can verify the right code is running
!git log --oneline -3

print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q Pillow tqdm
import sys
sys.path.insert(0, REPO_DIR)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU â€” Runtime > Change runtime type > T4 GPU')

## 2. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy, time, json
import matplotlib.pyplot as plt

from src.evolvable.machine_coaching import Policy, Rule, Literal, MCSymbolicModule
from src.evolvable.translator import Translator
from src.evolvable.dataset import create_experiment_data
from src.evolvable.evolution import EvolutionaryEngine, generate_offspring, summarize_evolution
import src.evolvable.evolution as evo_module

print('Imports OK.')

# Verify the updated generate_offspring is loaded (should return list for S+)
import inspect
from src.evolvable.evolution import mutate_Splus
_test_sig = inspect.signature(mutate_Splus)
# Quick smoke-test: S+ must return a list of 10
from src.evolvable.machine_coaching import Policy, MCSymbolicModule
_dummy_atoms = ['a1','a2','a3','a4']

class _DummyOrganism:
    atoms = _dummy_atoms
    symbolic = MCSymbolicModule(policy=Policy(), atoms=_dummy_atoms)
    def copy_organism(self): return self

_splus_out = mutate_Splus(_DummyOrganism(), _dummy_atoms)
assert isinstance(_splus_out, list) and len(_splus_out) == 10, f'S+ should return list of 10, got {_splus_out}'
print(f'Code version check OK — S+ generates {len(_splus_out)} offspring (expected 10)')
print('Imports OK.')


## 3. Hyperparameters

In [ ]:
NUM_ATOMS       = 4
NUM_RULES       = 4
POLICY_SEED     = 42
DATA_SEED       = 1042
TRAIN_SIZE      = 300
VAL_SIZE        = 100
TEST_SIZE       = 100
TRAIN_EPOCHS    = 8      # epochs per offspring per generation
BATCH_SIZE      = 64
LR              = 1e-3
MAX_GENERATIONS = 15
EARLY_STOP      = 0.95

atoms = [f'a{i+1}' for i in range(NUM_ATOMS)]
print(f'Atoms: {atoms}')
print(f'{MAX_GENERATIONS} generations x ~22+ offspring (S0x2, S+x20, S-x2*body) x {TRAIN_EPOCHS} epochs x {TRAIN_SIZE} samples')

## 4. PyTorch Organism

Replaces the pure-NumPy CNN with a PyTorch CNN that runs on GPU.  
The symbolic module, translator, and evolutionary logic are unchanged.

In [ ]:
class AtomCNN(nn.Module):
    """
    CNN that maps a 28x28 image to 2*num_atoms logits.
    For atom i: neurons [2*i, 2*i+1] = [P(positive), P(negative)].
    """
    def __init__(self, num_atoms: int):
        super().__init__()
        self.num_atoms = num_atoms
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)   # 28->26
        self.pool  = nn.MaxPool2d(2, 2)                # 26->13
        self.fc1   = nn.Linear(16 * 13 * 13, 64)
        self.fc2   = nn.Linear(64, 2 * num_atoms)

    def forward(self, x):
        """x: (N, 1, 28, 28) -> logits (N, 2*num_atoms)"""
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class TorchNeSyOrganism:
    """
    NeSy Organism backed by a PyTorch CNN instead of NumPy.
    Identical interface to NeSyOrganism â€” plugs into EvolutionaryEngine.
    """

    def __init__(self, atoms, policy=None, model=None):
        self.atoms     = list(atoms)
        self.num_atoms = len(atoms)
        self.symbolic  = MCSymbolicModule(policy=policy or Policy(), atoms=self.atoms)
        self.translator = Translator(self.atoms)
        self.model = model if model is not None else AtomCNN(self.num_atoms).to(device)

    # ------------------------------------------------------------------ #
    # Forward pass                                                         #
    # ------------------------------------------------------------------ #

    def _run_cnn(self, images_np):
        """
        images_np: (num_atoms, 1, 28, 28) numpy array for one sample.
        Returns: (num_atoms, 2*num_atoms) numpy logits, softmaxed per-atom.
        """
        with torch.no_grad():
            x = torch.FloatTensor(images_np).to(device)  # (num_atoms, 1, 28, 28)
            logits = self.model(x).cpu().numpy()          # (num_atoms, 2*num_atoms)

        # Build neural_output: for atom i use neurons [2i,2i+1] from image i
        neural_output = np.zeros(2 * self.num_atoms)
        for i in range(self.num_atoms):
            neural_output[2*i]   = logits[i, 2*i]
            neural_output[2*i+1] = logits[i, 2*i+1]

        # Softmax per atom pair
        probs = np.zeros(2 * self.num_atoms)
        for i in range(self.num_atoms):
            pair = neural_output[2*i:2*i+2]
            e = np.exp(pair - pair.max())
            probs[2*i:2*i+2] = e / (e.sum() + 1e-10)

        return probs

    def deduce(self, images):
        if images.ndim == 3:
            images = images[:, np.newaxis, :, :]  # (num_atoms,1,28,28)
        probs = self._run_cnn(images)
        atom_values = self.translator.neural_to_symbolic(probs)
        decision = self.symbolic.deduce(atom_values)
        return {'decision': decision, 'atom_values': atom_values,
                'probabilities': probs}

    # ------------------------------------------------------------------ #
    # Training (batched, GPU)                                              #
    # ------------------------------------------------------------------ #

    def train_epoch(self, dataset, learning_rate=LR, batch_size=BATCH_SIZE):
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate)

        N = len(dataset)
        perm = np.random.permutation(N)
        total_loss, proof_count, n_batches = 0.0, 0, 0

        for start in range(0, N, batch_size):
            idx = perm[start:start + batch_size]
            B   = len(idx)

            # Stack: (B*num_atoms, 1, 28, 28)
            imgs_list, labels = [], []
            for i in idx:
                s = dataset[i]
                imgs = s['images']
                if imgs.ndim == 3:
                    imgs = imgs[:, np.newaxis, :, :]
                imgs_list.append(imgs)
                labels.append(s['label'])

            imgs_t = torch.FloatTensor(
                np.concatenate(imgs_list, axis=0)
            ).to(device)  # (B*num_atoms, 1, 28, 28)

            optimizer.zero_grad()
            logits = self.model(imgs_t)  # (B*num_atoms, 2*num_atoms)

            grad = torch.zeros_like(logits)
            batch_loss = 0.0

            for j in range(B):
                proofs = self.symbolic.abduce(labels[j])
                if not proofs:
                    continue

                # Build neural_output for this sample
                sample_logits = logits[j*self.num_atoms:(j+1)*self.num_atoms]  # (num_atoms, 2*num_atoms)
                neural_out = np.zeros(2 * self.num_atoms)
                for i in range(self.num_atoms):
                    neural_out[2*i]   = sample_logits[i, 2*i].item()
                    neural_out[2*i+1] = sample_logits[i, 2*i+1].item()

                loss_val, grad_np = self.translator.compute_semantic_loss(neural_out, proofs)
                batch_loss += loss_val
                proof_count += 1

                # Inject gradient: only the relevant 2 neurons per atom-image
                for i in range(self.num_atoms):
                    row = j * self.num_atoms + i
                    grad[row, 2*i]   = float(grad_np[2*i])
                    grad[row, 2*i+1] = float(grad_np[2*i+1])

            if proof_count > 0:
                logits.backward(grad / B)
                optimizer.step()
                total_loss += batch_loss / B
                n_batches  += 1

        return {
            'avg_loss':   total_loss / max(n_batches, 1),
            'proof_rate': proof_count / max(N, 1),
        }

    # ------------------------------------------------------------------ #
    # Evaluation (batched)                                                 #
    # ------------------------------------------------------------------ #

    def evaluate(self, dataset):
        self.model.eval()
        correct, wrong, abstain = 0, 0, 0

        with torch.no_grad():
            for start in range(0, len(dataset), BATCH_SIZE):
                batch = dataset[start:start + BATCH_SIZE]
                for s in batch:
                    imgs = s['images']
                    if imgs.ndim == 3:
                        imgs = imgs[:, np.newaxis, :, :]
                    probs = self._run_cnn(imgs)
                    atom_values = self.translator.neural_to_symbolic(probs)
                    decision = self.symbolic.deduce(atom_values)
                    if decision is None:
                        abstain += 1
                    elif decision == s['label']:
                        correct += 1
                    else:
                        wrong += 1

        total = len(dataset)
        return {
            'correct':  correct  / max(total, 1),
            'wrong':    wrong    / max(total, 1),
            'abstain':  abstain  / max(total, 1),
            'total':    total,
        }

    # ------------------------------------------------------------------ #
    # Fitness                                                              #
    # ------------------------------------------------------------------ #

    def compute_relative_fitness(self, parent_results, own_results, labels):
        _SCORE = {
            ('correct','correct'):0, ('correct','wrong'):-1, ('correct','abstain'):-1,
            ('wrong','correct'):+1, ('wrong','wrong'):0,    ('wrong','abstain'):+1,
            ('abstain','correct'):+1,('abstain','wrong'):-1,('abstain','abstain'):0,
        }
        def cat(d, l): return 'abstain' if d is None else ('correct' if d==l else 'wrong')
        score = sum(_SCORE.get((cat(p,l), cat(o,l)), 0)
                    for p, o, l in zip(parent_results, own_results, labels))
        return score / max(len(labels), 1)

    # ------------------------------------------------------------------ #
    # Weight I/O (for Npw / Nrw mutations)                                #
    # ------------------------------------------------------------------ #

    def get_weights(self):
        return {k: v.cpu().numpy().copy() for k, v in self.model.state_dict().items()}

    def set_weights(self, weights):
        sd = {k: torch.FloatTensor(v) for k, v in weights.items()}
        self.model.load_state_dict(sd)
        self.model.to(device)

    def _reinit_weights(self):
        """Random weight reinitialization (Nrw mutation)."""
        new_model = AtomCNN(self.num_atoms).to(device)
        self.model.load_state_dict(new_model.state_dict())

    def copy_organism(self):
        new = TorchNeSyOrganism(atoms=list(self.atoms),
                                 policy=self.symbolic.policy.copy())
        new.set_weights(self.get_weights())
        return new


print('TorchNeSyOrganism defined.')

In [ ]:
# Monkey-patch the mutation functions so offspring are also Torch-based
import random

_orig_Nrw = evo_module.mutate_Nrw
_orig_Npw = evo_module.mutate_Npw

def _torch_Nrw(child):
    if isinstance(child, TorchNeSyOrganism):
        child._reinit_weights()
    else:
        _orig_Nrw(child)

def _torch_Npw(child, parent):
    if isinstance(child, TorchNeSyOrganism):
        child.set_weights(parent.get_weights())
    else:
        _orig_Npw(child, parent)

evo_module.mutate_Nrw = _torch_Nrw
evo_module.mutate_Npw = _torch_Npw
evo_module.NEURAL_MUTATIONS = {'Npw': _torch_Npw, 'Nrw': _torch_Nrw}

print('Evolution mutation functions patched for PyTorch.')

## 5. Generate Dataset

In [ ]:
print('Generating target policy and dataset...')
data = create_experiment_data(
    num_atoms=NUM_ATOMS,
    num_rules=NUM_RULES,
    policy_seed=POLICY_SEED,
    data_seed=DATA_SEED,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
)

target_policy = data['target_policy']
train_data    = data['train']
val_data      = data['val']
test_data     = data['test']

train_pos = sum(1 for s in train_data if s['label'])
val_pos   = sum(1 for s in val_data   if s['label'])

print(f'Target policy:\n{target_policy}')
print(f'Train: {len(train_data)} ({train_pos} pos / {len(train_data)-train_pos} neg)')
print(f'Val:   {len(val_data)}   ({val_pos} pos / {len(val_data)-val_pos} neg)')
print(f'Test:  {len(test_data)}')

## 6. Run Evolutionary Loop

We run the engine directly so we can pass a `TorchNeSyOrganism` as the initial organism.  
All 6 offspring per generation are also `TorchNeSyOrganism` (via `copy_organism`).

In [ ]:
engine = EvolutionaryEngine(
    atoms=atoms,
    train_epochs=TRAIN_EPOCHS,
    learning_rate=LR,
    max_generations=MAX_GENERATIONS,
    early_stop_accuracy=EARLY_STOP,
    verbose=True,
)

initial_organism = TorchNeSyOrganism(atoms=atoms)

t0 = time.time()
evo_result = engine.run(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    initial_organism=initial_organism,
)
elapsed = time.time() - t0
print(f'\nTotal time: {elapsed:.1f}s  ({elapsed/60:.1f} min)')

## 7. Results Summary

In [ ]:
summary = summarize_evolution(evo_result['history'])

print('=' * 55)
print('  EVOLVABLE POLICIES â€” RESULTS')
print('=' * 55)
print(f'  Generations run    : {summary["num_generations"]}')
print(f'  Initial accuracy   : {summary["initial_accuracy"]:.3f}')
print(f'  Final accuracy     : {summary["final_accuracy"]:.3f}')
print(f'  Best val accuracy  : {summary["best_accuracy"]:.3f}')
print(f'  Final wrong rate   : {summary["final_wrong_rate"]:.3f}')
print(f'  Final abstain rate : {summary["final_abstain_rate"]:.3f}')
print(f'  Selection counts   : {summary["selection_counts"]}')

if evo_result.get('final_test'):
    t = evo_result['final_test']
    print()
    print('  TEST SET:')
    print(f'    Correct  : {t["correct"]:.3f}')
    print(f'    Wrong    : {t["wrong"]:.3f}')
    print(f'    Abstain  : {t["abstain"]:.3f}')
print('=' * 55)
print()
print('Hidden target policy:')
print(target_policy)

## 8. Plots

In [ ]:
lineage = evo_result.get('lineage', [])
history = evo_result.get('history', [])

# Colour by symbolic mutation type (S0/S+/S-)
def _label_colour(lbl):
    if lbl.startswith('S0'): return '#95a5a6'
    if lbl.startswith('S+'): return '#2ecc71'
    if lbl.startswith('S-'): return '#e74c3c'
    return '#3498db'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: accuracy per generation
ax = axes[0]
if history:
    gens   = [h['generation']  for h in history]
    corr   = [h['parent_eval']['correct']  for h in history]
    wrong  = [h['parent_eval']['wrong']    for h in history]
    abst   = [h['parent_eval']['abstain']  for h in history]
    ax.plot(gens, corr,  color='#2ecc71', linewidth=2, marker='o', markersize=4, label='Correct')
    ax.plot(gens, wrong, color='#e74c3c', linewidth=2, marker='s', markersize=4, label='Wrong')
    ax.plot(gens, abst,  color='#95a5a6', linewidth=2, marker='^', markersize=4, label='Abstain')
ax.set_xlabel('Generation'); ax.set_ylabel('Fraction')
ax.set_title('Accuracy Across Generations')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(True, alpha=0.3)

# Right: fitness per generation (coloured by mutation type)
ax = axes[1]
if lineage:
    gens_l     = [g for g, _, _ in lineage]
    fitnesses  = [f for _, _, f in lineage]
    labels_l   = [l for _, l, _ in lineage]
    colours    = [_label_colour(l) for l in labels_l]
    ax.bar(gens_l, fitnesses, color=colours, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor='#95a5a6',label='S0'), Patch(facecolor='#2ecc71',label='S+'), Patch(facecolor='#e74c3c',label='S-')],
              fontsize=7)
ax.set_xlabel('Generation'); ax.set_ylabel('Relative Fitness')
ax.set_title('Mutation Fitness per Generation')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('evolvable_fitness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: evolvable_fitness.png')

## 9. Lineage & Paper Table

In [ ]:
if lineage:
    print(f'{"Gen":>4}  {"Mutation":<12}  {"Fitness":>8}')
    print('-' * 30)
    for gen, label, fitness in lineage:
        mark = ' +' if fitness > 0 else (' =' if fitness == 0 else ' -')
        print(f'{gen:>4}  {label:<12}  {fitness:>+8.3f}{mark}')

print()
print('='*58)
print('  PAPER TABLE')
print('='*58)
print(f'  Config: {NUM_ATOMS} atoms, {NUM_RULES} target rules, {MAX_GENERATIONS} max generations')
print(f'  Initial accuracy : {summary["initial_accuracy"]:.3f}')
print(f'  Final accuracy   : {summary["final_accuracy"]:.3f}')
print(f'  Best accuracy    : {summary["best_accuracy"]:.3f}')
print(f'  Generations run  : {summary["num_generations"]}')
if evo_result.get('final_test'):
    t = evo_result['final_test']
    print(f'  Test correct     : {t["correct"]:.3f}')
print('='*58)

## 10. Save & Download

In [ ]:
def to_python(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, list): return [to_python(v) for v in obj]
    return obj

results = {
    'config': {
        'num_atoms': NUM_ATOMS, 'num_rules': NUM_RULES,
        'policy_seed': POLICY_SEED, 'max_generations': MAX_GENERATIONS,
        'train_epochs': TRAIN_EPOCHS, 'train_size': TRAIN_SIZE,
    },
    'target_policy': str(target_policy),
    'summary': summary,
    'lineage': lineage,
    'final_test': evo_result.get('final_test'),
    'total_time': elapsed,
}

os.makedirs('results', exist_ok=True)
output_path = 'results/evolvable_torch_results.json'
with open(output_path, 'w') as f:
    json.dump(to_python(results), f, indent=2)
print(f'Saved {os.path.getsize(output_path)/1024:.1f} KB -> {output_path}')

try:
    from google.colab import files
    files.download(output_path)
    files.download('evolvable_fitness.png')
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab.')